# 00 Data: download, verify, build splits

Runs once. Safe to re-run after a disconnect: the download and each cleaned day are skipped when already present.

In [ ]:
# Setup: mount Drive and import the project code.
# Put the project folder (src/, configs/) at MyDrive/paper1/code, or set REPO_URL to a git remote.
from google.colab import drive
drive.mount("/content/drive")

import logging, subprocess, sys
REPO_URL = ""  # optional
CODE = "/content/drive/MyDrive/paper1/code"
if REPO_URL:
    CODE = "/content/zdssl"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=False)
    subprocess.run(["git", "-C", CODE, "pull"], check=True)
sys.path.insert(0, f"{CODE}/src")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)

from zdssl.config import config_hash, load_config, resolve_paths
cfg = load_config(f"{CODE}/configs/paper1.yaml")
paths = resolve_paths(cfg)
print("code:", CODE, "| results:", paths["runs"])


In [ ]:
# 1. Download the official CSVs (about 7 GB) to the Colab disk.
!pip -q install awscli
from zdssl.data import download_raw
download_raw(cfg)
!ls -lh "{cfg['paths']['raw_dir']}"


In [ ]:
# 2. Clean each day to parquet and verify the dataset. Stops with an error if the counts are wrong.
import json
from zdssl.data import ingest_all, verify_counts
day_stats = ingest_all(cfg)
summary = verify_counts(day_stats, cfg)
paths["ingest_stats"].parent.mkdir(parents=True, exist_ok=True)
paths["ingest_stats"].write_text(json.dumps({"summary": summary, "days": day_stats}, indent=2))
print("data lines incl. repeated headers:", summary["raw_rows"])
print("repeated header lines removed:   ", summary["header_rows"])
print("flows after cleaning:            ", summary["clean_rows"])
summary["label_counts"]


In [ ]:
# 3. Build the splits. Raises LeakageError if any invariant fails; nothing is saved in that case.
from zdssl.splits import build_splits, save_splits
frame, manifest = build_splits(cfg, day_stats)
save_splits(frame, manifest, paths["splits"], paths["manifest"])
print("saved:", paths["splits"])


In [ ]:
# 4. Numbers to send back / put in the paper (all also stored in manifest.json on Drive).
for key in ("split_sizes", "n_features_B", "dropped_constant_columns", "quarantine",
            "exact_duplicates_removed", "test_exact_duplicates_removed",
            "test_rows_removed_seen_in_historical",
            "validation_rows_sharing_a_train_feature_profile", "sample_class_counts",
            "leakage_checks"):
    print(f"\n{key}:\n  {manifest[key]}")
if manifest["quarantine"]["max_hour_of_day_seen"] <= 12:
    print("\nNOTE: timestamps use a 12-hour clock without AM/PM; the quarantine gap is a lower bound.")
print("\ncalibration benign flows:", manifest["split_class_counts"]["tune"][cfg["data"]["benign_label"]])
